# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and performing an initial analysis of the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://mlcroissant.org/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as an object)
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Explore the available record sets, fields, and their `@id` values.

**Note:** In Croissant datasets, each table or data source is modeled as a *record set*. Each field and column in every record set is uniquely identified by its `@id`.

In [ ]:
# List all available record sets by @id
print('Record sets in dataset:')
for record_set in dataset.metadata.record_sets:
    print(f"- @id: {record_set.id} | name: {record_set.name}")

# Show example: iterate one record from each record set (if any exist)
print('\nSample records and their fields for each record set:')
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord set: {record_set.name} (@id: {record_set.id})")
    try:
        records_iter = dataset.records(record_set=record_set.id)
        sample = next(records_iter)
        print('  Example record:')
        for k in sample:
            print(f'    Field @id: {k}')
        break
    except StopIteration:
        print('  [No records available]')
    except Exception as e:
        print(f'  Failed to load records: {e}')

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis. Always use the record set and field `@id` values found in Section 2.

In [ ]:
# Get all record set @id values
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'Record set {record_set_id}:')
        print(df.columns.tolist())
        display(df.head(2))
    else:
        print(f'Record set {record_set_id}: [No records to extract]')

# For demonstration, select the first non-empty record set
if dataframes:
    picked_record_set_id = next(iter(dataframes))  # Use this value for later analysis
    print(f"\nChosen record set for demonstration: {picked_record_set_id}")
    display(dataframes[picked_record_set_id].head(5))
else:
    print('No tabular data found in the Croissant record sets.')

## 4. Exploratory Data Analysis (EDA)
We'll apply some typical data processing and exploration steps: filtering numeric fields, normalizing, grouping, and basic statistics. For demonstration, we select a numeric field and group field by their `@id` (replace below if needed).

In [ ]:
import numpy as np

# Identify available DataFrame and columns (field @id)
if dataframes:
    df = dataframes[picked_record_set_id]
    columns = df.columns.tolist()
    # Heuristic: Try to find a numeric column (int/float) for demo
    numeric_field = None
    for col in columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Try coercing to numeric if possible
        for col in columns:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notnull().sum() > 0:
                numeric_field = col
                df[col] = coerced
                break
    print(f'Numeric field chosen: {numeric_field}')

    # Choose a grouping field (@id) if possible
    candidate_cats = [col for col in columns if df[col].dtype==object and df[col].nunique() > 1 and df[col].nunique()<25]
    group_field = candidate_cats[0] if candidate_cats else None
    print(f'Group field chosen: {group_field}')

    # Apply a filter if possible
    if numeric_field:
        thresh = df[numeric_field].mean() if np.issubdtype(df[numeric_field].dtype, np.number) else 0
        filtered_df = df[df[numeric_field] > thresh]
        print(f"\nFiltered records with {numeric_field} > {thresh:.2f} (using field @id):")
        display(filtered_df.head())

        # Add normalized column
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/
            filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print('No numeric field detected to analyze.')
else:
    print('No data available for EDA.')

## 5. Visualization
Let's visualize the distribution of the chosen numeric field and relationship between numeric and group fields if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    fig, ax = plt.subplots(1, 2 if group_field else 1, figsize=(14, 5))
    
    # Histogram
    sns.histplot(df[numeric_field].dropna(), kde=True, ax=ax[0] if group_field else ax)
    ax[0].set_title(f"Distribution of {numeric_field} (@id)")
    ax[0].set_xlabel(numeric_field)

    # Boxplot by group
    if group_field and group_field in df.columns:
        sns.boxplot(x=group_field, y=numeric_field, data=df, ax=ax[1], whis=1.5)
        ax[1].set_title(f"{numeric_field} by {group_field} (@id)")
        for label in ax[1].get_xticklabels():
            label.set_rotation(45)

    plt.tight_layout()
    plt.show()
else:
    print('No numeric data available for visualization.')

## 6. Conclusion
In this notebook, we:

- Loaded the dataset via the Croissant schema and explored its metadata.
- Listed available record sets and demonstrated extraction of tabular records by referencing their `@id` values.
- Performed basic exploratory data analysis on selected numeric and group fields (identified by `@id`).
- Created example visualizations to summarize numeric distributions and group-wise comparisons.

This workflow can be adapted for deeper analysis or integration with machine learning pipelines. 

**Important:** When processing Croissant datasets, always reference record sets, fields, and columns using their `@id` to ensure reproducibility and clarity.

For more, visit the [mlcroissant documentation](https://mlcroissant.org/) or explore the [FAIR² data portal](https://sen.science/doi/10.71728/senscience.y7m0-f273).